In this notebook we'll compare the results of three Promotions to determine which was the most effective, beginning with standard frequentist techniques and concluding with a Bayesian approach. Our frequentist measures will conclude that Promotions 1 and 3 perform better than Promotion 2, but we are unable to distinguish a significant difference between the two. A Bayesian approach will demonstrate that Promotion 1 performs significantly better than Promotion 3.

The promotions in the small and medium market sizes are close to normally distributed, but the large market size takes on a bimodal distribution. Using a mixture model allows us to model each component of the large market size and reveal that Promotion 1 has the largest mean of each component. There is a significant difference in the mixture probabilities between the promotions, however, which warrants further investigation.

In [ ]:
!pip install pymc -q
!pip install scikit_posthocs -q

In [ ]:
import arviz
import matplotlib.pyplot as plt
import numpy
import pandas
import plotly.express as px
import pymc
import scikit_posthocs
import scipy.stats as stats
import statsmodels

category_orders = dict(MarketSize=["Small", "Medium", "Large"], Promotion=["1", "2", "3"])

numpy.random.seed(4435)

<h2>Exploratory Analysis</h2>

Let's open our dataset and look at the first few rows

In [ ]:
df = pandas.read_csv("/kaggle/input/fast-food-marketing-campaign-ab-test/WA_Marketing-Campaign.csv")
df.head()

and look at some metadata about our columns

In [ ]:
df.info()

We don't have any missing values. We also don't have any duplicate entries

In [ ]:
df.duplicated().sum()

and none our locations ran multiple promotions.

In [ ]:
df.groupby("LocationID")["Promotion"].nunique().nunique()

Let's cast the promotion type to a string to make plotting a bit easier:

In [ ]:
df["Promotion"] = df["Promotion"].astype("str")

Let's plot our sample distribution by promotion and market size.

In [ ]:
px.bar(
    df.groupby(["MarketSize", "Promotion"]).size().rename("Count").reset_index(),
    x="MarketSize",
    y="Count",
    color="Promotion",
    barmode="group",
    category_orders=category_orders,
)

A medium market size is most common, with about 5x more samples than our small market and about 2x the number as our large market size. The promotion sizes appear to be fairly well distributed within each market.

Let's visualize our sales data by market size and promotion.

In [ ]:
px.box(
    df,
    x="MarketSize",
    y="SalesInThousands",
    color="Promotion",
    category_orders=category_orders
)

It looks like we tend to get more sales from our large marketplace, while our small marketplaces tend to slightly outperform our medium markets. Our second promotion type performs a bit worse than the 1st and 3rd types, which look to be fairly comparable in their sales.

Let's view a histogram of our sales data. We'll split the data by market size and promotion type:

In [ ]:
px.histogram(
    df,
    x="SalesInThousands",
    facet_row="Promotion",
    facet_col="MarketSize",
    category_orders=category_orders,
    height=600,
    width=800
)

Our large market size appears to have a bimodal distribution while the other two look like they could be normally distributed.

Let's look at a breakdown by the age of the store. It looks like the promotions are fairly well evenly distributed.

In [ ]:
px.bar(
    df.groupby(["AgeOfStore", "Promotion"]).size().rename("Count").reset_index(),
    x="AgeOfStore",
    y="Count",
    color="Promotion",
    barmode="stack",
    category_orders=category_orders
)

Looking at our sales dependency on the age of the store, there isn't too much of an age dependency. Promotion 2 seems to come out a bit worse on average, like noted before.

In [ ]:
px.scatter(df, x="AgeOfStore", y="SalesInThousands", color="Promotion", category_orders=category_orders)

Let's examine our sales by week to see if we have any evidence of a novelty effect taking place with a large spike in sales early on followed by a decline at the end. Our sales look like they are consistently distributed across the weeks.

In [ ]:
px.histogram(
    df,
    x="SalesInThousands",
    facet_col="Promotion",
    facet_row="week",
    color="MarketSize",
    category_orders=category_orders,
)

## Statistical Analysis

### ANOVA

ANOVA (Analysis of Variance) is a set of methods used to assess whether the means across multiple groups are different. There are three assumptions ANOVA relies on:
1. Data is independent
2. Responses are normally distributed
3. Groups share the same variance

We have demonstrated that there is no crossover in the promotion groups, so the first assumption is met. Let's check the other two.

#### Normality

Normality is often examined graphically with a [qqplot](https://www.r-bloggers.com/2020/08/q-q-plots-and-worm-plots-from-scratch/), which is shown below where we split our dataset by promotion and market size:

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def se(z, n):
    p = stats.norm.cdf(z)
    d = stats.norm.pdf(z)
    return (p*(1-p)/n)**0.5 / d
    
def calc_qq(data):
    quantile = (data.rank() - 0.5) / len(data)
    mean = data.mean()
    std = data.std()

    return stats.norm.ppf(quantile)

def theoretical_line(data, zmin=-3, zmax=3):
    z_norm = stats.norm.ppf([0.75, 0.25])
    iqr = data.quantile(0.75) - data.quantile(0.25)
    median = data.median()

    # Trend line
    def foo(z):
        return median + (iqr / 1.349) * z

    return foo

fig = make_subplots(
    rows=len(category_orders["MarketSize"]),
    cols=len(category_orders["Promotion"]),
    row_titles=[f"MarketSize = {x}" for x in category_orders["MarketSize"]],
    column_titles=[f"Promotion={x}" for x in category_orders["Promotion"]],
    shared_xaxes=True,
    shared_yaxes=True,
)

for i,market in enumerate(category_orders["MarketSize"]):
    for j,promotion in enumerate(category_orders["Promotion"]):
        data = df.query(f"MarketSize == '{market}' & Promotion == '{promotion}'")["SalesInThousands"]
        znorm = calc_qq(data)
        trendline = theoretical_line(data)
        z_trend = numpy.linspace(znorm.min(), znorm.max(), 100)
        y_trend = trendline(z_trend)
        error = se(z_trend, len(znorm)) * data.std(ddof=1)
                
        # Trend Line
        fig.add_trace(
            go.Scatter(
                name = "Normal",
                x = z_trend,
                y = y_trend,
                mode = "lines",
                marker = dict(color="#444"),
                showlegend=False,
            ),
            row = i+1,
            col = j+1
        )

        # Confidence Interval
        fig.add_trace(
            go.Scatter(
                name = "Upper Bound",
                x = z_trend,
                y = y_trend+2*error,
                mode = "lines",
                marker = dict(color="#444"),
                showlegend = False,
            ),
            row = i + 1,
            col = j + 1
        )

        fig.add_trace(
            go.Scatter(
                name = "Lower Bound",
                x = z_trend,
                y = y_trend-2*error,
                mode = "lines",
                marker = dict(color="#444"),
                fillcolor = "rgba(68,68,68,0.3)",
                fill = "tonexty",
                showlegend = False
            ),
            row = i + 1,
            col = j + 1
        )
        
        # Observations
        fig.add_trace(
            go.Scatter(
                x = znorm,
                y = data,
                mode = "markers",
                showlegend=False,
                marker=dict(color=px.colors.qualitative.Plotly[i+3*j])
            ),
            row = i+1,
            col = j+1,
        )

        if i == 2:
            fig.update_xaxes(title_text="Normal Theoretical Quantiles", row=i+1, col=j+1)
        if j == 0:
            fig.update_yaxes(title_text="Sales in Thousands", row=i+1, col=j+1)

fig.update_layout(width=1000, height=800)
fig.show()

Our sales for the small and medium market sizes look to be normally distributed for each promotion, but our distributions for the large market size look non-normal.

To evaluate how well the distributions match a normal distribution we can make use of the Shapiro-Wilk test. Our hypothesis are:
- Null hypothesis: data is normally distributed
- Alternative: data is non-normally distributed

In [ ]:
df.groupby(["MarketSize", "Promotion"]).agg({
    "SalesInThousands": [
        ("Count", "count"),
        ("Mean", "mean"),
        ("STD", "std"),
        ("Shapiro-Wilk (p)", lambda x: stats.shapiro(x)[1]),
    ]
})

Using $p=0.05$ we can't reject the null hypothesis that are distribution is normal for the medium and small markets across the different promotions, but we can for the large market size.

#### Variance

To examine the variances, let's make use of Levene's test as it can work with non-normal distributions. Our hypothesis are
- Null hypothesis: variances are the same across groups
- Alternative hypothesis: at least one group has a different variance

In [ ]:
levene_test = {}
for market in category_orders["MarketSize"]:
    groups = []
    for promotion in category_orders["Promotion"]:
        group = df.query(f"MarketSize == '{market}' and Promotion == '{promotion}'")
        groups.append(group["SalesInThousands"])
    levene_test[market] = stats.levene(*groups)[1]

for k,v in levene_test.items():
    print("Market Size: {:>6}, p-value: {:.2f}".format(k,v))

Using a significance level of $p=0.05$, then we can't reject the null hypothesis that our distributions have the same variance.

#### Tukey's test

Our Large market size violates the ANOVA assumptions, but appropriate ANOVA metrics can be applied to the small and medium markets. Let's go ahead and use Tukey's test to compare the means across each of the promotions in these markets:

In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey_test = {}
for market in category_orders["MarketSize"]:
    if market == "Large":
        continue

    group = df.query(f"MarketSize == '{market}'")
    tukey_test[market] = pairwise_tukeyhsd(
        endog=group["SalesInThousands"],
        groups=group["Promotion"],
        alpha=0.05
    )

for k,v in tukey_test.items():
    print("Market Size: {}, {}\n".format(k,v))

Tukey's test indicates that we can reject the notion of equal sales across our three promotions, with Promotions 1 and 3 both having a larger mean than Promotion 2 for each market size. Our p value prevents us from differentiating between Promotions 1 and 3, so we can't reject the null hypothesis that they perform equally well.

#### Kruskal-Wallis + Canover Tests

Evaluating our Large market requires the use of a non-parameteric test. Since we have more than two groups, we can make use of the Kruskal-Wallis (KW) test which compares the medians across groups and serves as a non-parametric alternative to ANOVA.

In [ ]:
kw_test = {}
for market in category_orders["MarketSize"]:
    groups = []
    for promotion in category_orders["Promotion"]:
        group = df.query(f"MarketSize == '{market}' and Promotion == '{promotion}'")
        groups.append(group["SalesInThousands"])
    kw_test[market] = stats.kruskal(*groups)[1]

for k,v in kw_test.items():
    print("Market Size: {:>6}, p-value: {:.2e}".format(k,v))

From the KW test we can reject the null hypothesis that the medians across each promotion are the same. To perform a pairwise test we can use Canover's test:

In [ ]:
conover_test = {}
for market in category_orders["MarketSize"]:
    group = df.query(f"MarketSize == '{market}'")
    conover_test[market] = scikit_posthocs.posthoc_conover(
        group,
        val_col="SalesInThousands",
        group_col="Promotion",
        p_adjust="holm",
    )

with pandas.option_context("display.float_format", "{:.2f}".format):
    for k,v in conover_test.items():
        print("Market Size: {}\n{}\n".format(k,v))

In all three market sizes we can reject the null hypothesis that Promotion 1-2 and 2-3 have the same medians, but can't reject the hypothesis that Promotions 1-3 have the same median.

We therefore recommend the use of Promotion 1 or 3, which give the same performance to within our uncertainty.

### Bayesian

In this section we'll take a Bayesian approach towards analyzing our data. Since the sales data in our small and medium markets appear to be consistent with a normal distribution while the large market looks bimodal, we'll take two different approaches in our analysis.

#### Small and Medium Markets

Since our sales data for the small and medium markets is consistent with a normal distribution while having similar variances, let's apply Bayesian modeling with these assumptions.

We'll begin by extracting these two markets from our dataset and create variables to represent the market and promotion type.

In [ ]:
data = df.query("MarketSize != 'Large'")

promo_idx, promo = pandas.factorize(data["Promotion"], sort=True)
market_idx, market = pandas.factorize(data["MarketSize"], sort=True)

Let's make our priors consistent across promotion type, but allow for a difference between market sizes. We'll use a normal prior for our sales and a truncated normal for our standard deviation to avoid negative values.

In [ ]:
coords = dict(
    promo=promo,
    market=market
)

with pymc.Model(coords=coords) as model:

    # =============
    # Data indecies
    # =============
    market_idx = pymc.Data("market_idx", market_idx, mutable=False, dims="samples")
    promo_idx = pymc.Data("promo_idx", promo_idx, mutable=False, dims="samples")
    
    # ==================
    # Market size priors
    # ==================
    mu_market = pymc.Normal(
        "mu_market",
        mu=data["SalesInThousands"].mean(),
        sigma=50,
        dims="market",
    )
    
    sigma_market = pymc.TruncatedNormal(
        "std_market",
        mu = data["SalesInThousands"].std(ddof=1),
        sigma=50,
        dims="market",
        lower=0
    )
    
    # ===============================
    # Promotion priors by market size
    # ===============================
    mu = pymc.TruncatedNormal(
        "mu",
        mu=mu_market,
        sigma=50,
        dims=("promo", "market"),
        lower=0
    )

    sigma = pymc.TruncatedNormal(
        "sigma",
        mu=sigma_market,
        sigma=50,
        dims=("promo", "market"),
        lower=0
    )

    # ============
    # Observations
    # ============
    y = pymc.Normal(
        "sales",
        mu[promo_idx,market_idx],
        sigma = sigma[promo_idx,market_idx],
        observed = data["SalesInThousands"],
        dims="samples",
    )

pymc.model_to_graphviz(model)

In [ ]:
with model:
    trace = pymc.sample(1000, tune=1000)

If we check the posterior prediction on our sales data, we'll find that our model does a good job at reproducing our observations.

In [ ]:
draws_posterior = pymc.sample_posterior_predictive(trace, model=model)

ax = arviz.plot_ppc(draws_posterior, figsize=(12,8));
ax.set_xlabel("Sales in Thousands");

Let's look at the posterior distributions for our parameters.

In [ ]:
arviz.plot_trace(trace, var_names=["mu", "sigma"], legend=True, figsize=(16,8))
plt.tight_layout()

In [ ]:
arviz.plot_posterior(trace, var_names=["mu", "sigma"], grid=(6,2), hdi_prob=0.95);

Our second promotion performs considerably worse than Promotions 1 and 3. Promotion 1 appears to perform a bit better than Promotion 3 in our medium markets while the two are more comparable in the small markets.

Let's do some analysis on our posterior distributions for Promotions 1 and 3 to better compare their performance. We'll find the posterior distribution of the offset in the mean number of sales as well as the fraction of times that Promotion 1 has a higher mean that Promotion 3:

In [ ]:
medium1 = trace.posterior.sel(market="Medium", promo="1")["mu"]
medium3 = trace.posterior.sel(market="Medium", promo="3")["mu"]
small1 = trace.posterior.sel(market="Small", promo="1")["mu"]
small3 = trace.posterior.sel(market="Small", promo="3")["mu"]

# Differences
dmedium = (medium1 - medium3).values.reshape(-1)
dsmall = (small1 - small3).values.reshape(-1)

# Confidence levels (95%)
cl_medium = numpy.quantile(dmedium, 0.05)
cu_medium = numpy.quantile(dmedium, 0.95)

cl_small = numpy.quantile(dsmall, 0.05)
cu_small = numpy.quantile(dsmall, 0.95)

In [ ]:
fig = px.histogram(dsmall, marginal="box")
fig.add_vline(x=cl_small)
fig.add_vline(x=cu_small)
fig.update_layout(xaxis_title="Mean Sales In Thousands, Promotion 1 - 3", title="Small Markets")
fig.update_layout(showlegend=False)
fig.show()

fig = px.histogram(dmedium, marginal="box")
fig.add_vline(x=cl_medium)
fig.add_vline(x=cu_medium)
fig.update_layout(xaxis_title="Mean Sales In Thousands, Promotion 1 - 3", title="Medium Markets")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
print("Small:  Promotion 1 > Promotion 3 (%): ", (dsmall > 0).mean())
print("Medium: Promotion 1 > Promotion 3 (%): ", (dmedium > 0).mean())

There is about a 65% chance that Promotion 1 results in more sales than Promotion 3 in small markets and a 97% chance in medium markets.

We therefore recommend Promotion 1 as the optimal strategy in small and medium markets.

### Large Markets

The sales in our large market sizes are bimodally distributed, so in this part we'll make use a mixture model to analyze the data.

We'll assume that our sales data is generate by a mixture of two Gaussian components. We'll use a common mean and standard deviation prior for our promotions while a Dirichlet prior will be used for the mixture weights. We can create our model as follows:

In [ ]:
data = df.query("MarketSize == 'Large'")
promo_idx, promo = pandas.factorize(data["Promotion"], sort=True)

with pymc.Model(coords=dict(promo=promo, component=["lower", "upper"])) as large_model:

    # =============
    # Data indecies
    # =============
    promo_idx = pymc.Data("promo_idx", promo_idx, mutable=False, dims="samples")
  
    # =============
    # Market Priors
    # =============
    mu_market = pymc.Normal(
        "mu_market",
        mu = [60, 90],
        sigma = 15,
        transform = pymc.distributions.transforms.ordered
    )

    sigma_market = pymc.HalfNormal(
        "sigma_market",
        sigma = [50,50],
    )
    
    # ================
    # Promotion Priors
    # ================
    mu = pymc.Normal(
        "mu",
        mu = mu_market,
        sigma = 50,
        shape = (3,2),
    )
    
    sigma = pymc.HalfNormal(
        "sigma",
        sigma = sigma_market,
        shape = (3,2),
    )
    
    weights = pymc.Dirichlet("w", numpy.ones((3,2)))
    
    # ========
    # Fit data
    # ========
    pymc.NormalMixture(
        "sales",
        w=weights[promo_idx],
        mu=mu[promo_idx],
        sigma = sigma[promo_idx],
        observed=data["SalesInThousands"]
    )

pymc.model_to_graphviz(large_model)

In [ ]:
with large_model:
    trace_large = pymc.sample(1000, tune=1000, random_seed=10, chains=3)

Our model does a fairly good job when examining the posterior distribution on our sales data.

In [ ]:
draws_posterior = pymc.sample_posterior_predictive(trace_large, model=large_model)

ax = arviz.plot_ppc(draws_posterior, figsize=(12,8));
ax.set_xlabel("Sales in Thousands");

Let's look at the posterior distribution on our mean, standard deviations, and weights:

In [ ]:
arviz.plot_trace(trace_large, var_names=["mu", "sigma", "w"], legend=True, figsize=(16, 12))
plt.tight_layout()

Promotion 1 has the largest means of each component by a significant margin, with Promotion 3 coming in second. 

In terms of the mixture probabilities (weights), Promotion 1 was close to a 50%-50% split between the two mixtures, Promotion 2 has a 2/3 split favoring the smaller sales component, and Promotion 3 has about 3/4 of the samples occurring in the larger sales component. The component weights could be a reflection of the sample population rather than the effectiveness of the promotion, so it would be worth looking into whether our samples are truely independently sampled or if there is something like a geographical variable lurking in our large market data that is accounting for these different weights.

Assuming our weights are unrelated to the effectiveness of the promotion, then Promotion 1 is also recommended for the large markets.

## Conclusions

We took both a Frequentist and Bayesian approaches towards finding the optimal promotion strategy within the context of A/B/C testing. Our Frequentist methods suggest using either Promotion 1 or 3 is the optimal choice. Our Bayesian analysis, however, has shown that Promotion 1 outperforms Promotion 3 by a significant enough margin that we can conclude Promotion 1 is the most effective strategy of the three candidates.